In [ ]:
import os
import glob
import re
import random
import gc
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
import nltk
from tqdm import tqdm

from ai4bharat.transliteration import XlitEngine

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

nltk.download('wordnet', quiet=True)

In [ ]:
data_dir = "native"
out_dir = "outputs_backtransliteration/"
os.makedirs(out_dir, exist_ok=True)

model_name = "google/madlad400-10b-mt"  # "google/madlad400-3b-mt", "google/madlad400-7b-mt", "google/madlad400-10b-mt", 
device = "cuda" if torch.cuda.is_available() else "cpu"
output_path = f"{out_dir}{model_name.split('/')[1]}_mt_evaluation_transliteration_results_I.csv"

In [ ]:
BATCH_SIZE = 32  

LANG_MAP = {
    "Assamese": {"madlad": "as", "xlit": "as"},
    "Manipuri": {"madlad": "mni", "xlit": "mni"},       
    "Meitei-Mayek": {"madlad": "mni", "xlit": "mni"},
    "Bodo": {"madlad": "brx", "xlit": "brx"},
}

bleu_metric = evaluate.load("bleu")
chrf_metric = evaluate.load("chrf")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")
bertscore_metric = evaluate.load("bertscore")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, 
    use_safetensors=True, 
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True
).to(device)

if hasattr(torch, "compile") and device == "cuda":
    print("Compiling model for faster inference execution...")
    model = torch.compile(model)

In [ ]:
def determine_lang_info(filename):
    for lang, config in LANG_MAP.items():
        if lang.lower() in filename.lower():
            return lang, config
    return None, None

def detect_columns(df, filename):
    cols = [str(c).strip() for c in df.columns]    
    is_numeric_header = all(c.isdigit() for c in cols)
    
    if is_numeric_header:
        print(f" No named headers detected (numeric headers found). Using filename heuristics...")
        filename_lower = filename.lower()
        lang_name, _ = determine_lang_info(filename)
        
        if lang_name and filename_lower.find('eng') < filename_lower.find(lang_name.lower()):
            en_col, src_col = df.columns[0], df.columns[1]
        else:
            src_col, en_col = df.columns[0], df.columns[1]
            
        return src_col, en_col

    df.columns = cols  
    if len(cols) < 2:
        raise ValueError("File does not contain at least 2 columns.")
        
    en_indicators = ['english', 'en', 'eng', 'English']
    en_col = None
    for col in cols:
        if col.lower() in en_indicators:
            en_col = col
            break
    if not en_col:
        for col in cols:
            if 'eng' in col.lower() or col.lower() == 'en':
                en_col = col
                break
                
    if not en_col:
        print(" Unclear column names. Falling back to character distribution analysis...")
        ascii_scores = {}
        for col in cols:
            sample = df[col].dropna().head(100).astype(str)
            if sample.empty:
                ascii_scores[col] = 0
                continue
            total_chars = sum(len(s) for s in sample)
            ascii_chars = sum(len(re.findall(r'[A-Za-z0-9\s.,!?]', s)) for s in sample)
            ascii_scores[col] = ascii_chars / total_chars if total_chars > 0 else 0
        en_col = max(ascii_scores, key=ascii_scores.get)
        
    other_cols = [c for c in cols if c != en_col]
    src_col = other_cols[0] 
    return src_col, en_col

def run_transliteration(texts, xlit_code, batch_size=32):
    if not xlit_code:
        return texts
    
    print(f"-> Transliterating native text to Latin script via AI4Bharat using code: '{xlit_code}'")
    try:
        import argparse
        import inspect
        if hasattr(torch, "serialization") and hasattr(torch.serialization, "add_safe_globals"):
            torch.serialization.add_safe_globals([argparse.Namespace])
            
        e = XlitEngine(beam_width=4, rescore=False, src_script_type="indic")
        transliterated_texts = []
        
        translit_func = getattr(e, "translit_sentence", None) or getattr(e, "transliterate_sentence", None)
        if translit_func is None:
            raise AttributeError("Could not locate a valid transliteration method on XlitEngine.")
            
        sig = inspect.signature(translit_func)
        params = sig.parameters
        
        for i in tqdm(range(0, len(texts), batch_size), desc="Transliterating Blocks"):
            batch = texts[i:i + batch_size]
            
            if 'lang_code' in params:
                out_batch = [translit_func(text, lang_code=xlit_code) for text in batch]
            elif 'lang' in params:
                out_batch = [translit_func(text, lang=xlit_code) for text in batch]
            else:
                out_batch = [translit_func(text) for text in batch]
                
            transliterated_texts.extend(out_batch)
            
        return transliterated_texts
        
    except Exception as ex:
        print(f" [Error] Transliteration engine failed: {ex}. Using original text.")
        return texts
        
def translate_batch_generator(texts, tgt_code="en", batch_size=32):
    prefix = f"<2{tgt_code}> "
    for i in range(0, len(texts), batch_size):
        batch = [prefix + str(text) for text in texts[i:i+batch_size]]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.inference_mode():
            generated_tokens = model.generate(**inputs, max_length=256, num_beams=1, do_sample=False, early_stopping=True)
        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        yield decoded
        
def save_incremental_results(results_dict, out_path):
    if not results_dict:
        return
    res_df = pd.DataFrame.from_dict(results_dict, orient='index')
    if len(res_df) > 1:
        calc_df = res_df.drop('OVERALL (Macro Avg)', errors='ignore')
        overall_avg = calc_df.mean(numeric_only=True).round(2)
        res_df.loc['OVERALL (Macro Avg)'] = overall_avg
        
    res_df.to_csv(out_path)
    print(f"\n--- Live metrics written back to: {out_path} ---")

In [ ]:
all_files = glob.glob(os.path.join(data_dir, "*.csv")) + glob.glob(os.path.join(data_dir, "*.xlsx"))
language_results = {}

print(f"Found {len(all_files)} files to process.")

for file_path in all_files:
    filename = os.path.basename(file_path)
    lang_name, config = determine_lang_info(filename)
    
    if not lang_name:
        print(f"Skipping {filename}: Could not determine target language grouping.")
        continue
        
    src_code = config["madlad"]
    xlit_code = config["xlit"]
    
    if not src_code:
        print(f"Skipping {filename}: Language '{lang_name}' is not supported by MADLAD-400.")
        continue   
    print(f"\nProcessing {lang_name} ({filename})...")   
    try:
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path)
        else:
            df = pd.read_excel(file_path)     
        src_col, en_col = detect_columns(df, filename)
        print(f"-> Mapped input columns: [Source: '{src_col}'] to [English Target: '{en_col}']")    
        df = df[[src_col, en_col]].dropna().astype(str)
        
        raw_src_sentences = df[src_col].tolist()
        ref_sentences = df[en_col].tolist()
        if not raw_src_sentences:
            print(f" Empty dataset for {filename} after scrubbing null elements.")
            continue 
            
        src_sentences = run_transliteration(raw_src_sentences, xlit_code, batch_size=BATCH_SIZE)
        
        print(f"Translating {len(src_sentences)} sentences using optimized batching...")
        hyp_sentences = []
        
        for batch_hyp in tqdm(translate_batch_generator(src_sentences, tgt_code="en", batch_size=BATCH_SIZE), 
                              total=(len(src_sentences) + (BATCH_SIZE - 1)) // BATCH_SIZE, desc="Batches"):
            hyp_sentences.extend(batch_hyp)
                
        prediction_df = df.copy()
        prediction_df["Prediction"] = hyp_sentences
        
        prediction_output_path = f"{os.path.splitext(filename)[0]}_transliterated_predictions.csv"
        prediction_df.to_csv(prediction_output_path, index=False)
        print(f"Predictions saved to: {prediction_output_path}")  
        print("Computing metrics safely...")
        bleu_ref = [[ref] for ref in ref_sentences]
        
        scores = {"BLEU": None, "chrF": None, "ROUGE-1": None, "ROUGE-L": None, "METEOR": None, "BERTScore-F1": None}
        
        try:
            bleu = bleu_metric.compute(predictions=hyp_sentences, references=bleu_ref)
            scores["BLEU"] = round(bleu['bleu'] * 100, 2)
        except Exception as me:
            print(f"  [Error] Failed calculating BLEU: {me}")
            
        try:
            chrf = chrf_metric.compute(predictions=hyp_sentences, references=bleu_ref)
            scores["chrF"] = round(chrf['score'], 2)
        except Exception as me:
            print(f"  [Error] Failed calculating chrF: {me}")
            
        try:
            rouge = rouge_metric.compute(predictions=hyp_sentences, references=ref_sentences)
            scores["ROUGE-1"] = round(rouge['rouge1'] * 100, 2)
            scores["ROUGE-L"] = round(rouge['rougeL'] * 100, 2)
        except Exception as me:
            print(f"  [Error] Failed calculating ROUGE: {me}")
            
        try:
            meteor = meteor_metric.compute(predictions=hyp_sentences, references=ref_sentences)
            scores["METEOR"] = round(meteor['meteor'] * 100, 2)
        except Exception as me:
            print(f"  [Error] Failed calculating METEOR: {me}")
            
        try:
            bertscore = bertscore_metric.compute(
                predictions=hyp_sentences, 
                references=ref_sentences, 
                lang="en", 
                batch_size=BATCH_SIZE, 
                device=device
            )
            scores["BERTScore-F1"] = round(sum(bertscore['f1']) / len(bertscore['f1']) * 100, 2)
        except Exception as me:
            print(f"  [Error] Failed calculating BERTScore: {me}")
            
        language_results[lang_name] = scores
        save_incremental_results(language_results, output_path)
        
    except Exception as e:
        print(f" Critical error occurred while processing {lang_name} ({filename}): {e}")
    finally:
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()
        print("Moving to next language file...")

if language_results:
    final_df = pd.DataFrame.from_dict(language_results, orient='index')
    calc_df = final_df.drop('OVERALL (Macro Avg)', errors='ignore')
    overall_avg = calc_df.mean(numeric_only=True).round(2)
    final_df.loc['OVERALL (Macro Avg)'] = overall_avg  
    print("\n" + "="*60)
    print(" FINAL EVALUATION SUMMARY ")
    print("="*60)
    print(final_df.to_markdown())
else:
    print("No languages were successfully processed.")